# H1 — Layer-dependent intervention

**This notebook runs the complete H1 experiment, including its own controls, tuning and report.**
It can run concurrently with the other two notebooks when each has a separate GPU runtime.
All three use the same **54 landmarks**, with **18 calibration + 36 confirmation** and **up to 50
eligible distinct photographs per landmark**. Smaller landmarks retain all eligible views; none are
replaced or padded. All methods use identical images for each landmark, and tables report actual counts.
Each has a separate output folder, checkpoints and logs. The 79 final holdout landmarks are untouched.

**Question:** Does adapting the coefficient to each layer improve reconstruction over one global scalar?

**Method and decision variables:** Use eta_l = min(100000, alpha * norm(T_l[M]) / max(norm(g_l[M]), 1e-8)). Tune alpha in {0.01, 0.02}; scalar controls tune eta in {100, 1000}. Keep the reference objective, all-token update and accept/revert rule.

Pick settings using mean calibration normalized Chamfer distance only; freeze before confirmation.
Primary outcome: the paired confirmation-landmark difference, structured − global scalar (negative is better).
Also report the no-intervention comparison, completeness, failures and effective-strength traces.

**Budget:** at most 198 scene/condition jobs plus reference checks. An eight-hour session limit
permits resume; it is not a measured completion estimate. Each GPU must pass the 50-image memory pilot.
The three notebooks repeat controls and some preparation; total GPU work
is higher than the single shared-runner design. Parallel speedup has not been measured.

**Shared masks:** first finish `Point3_00_Prepare_Masks_UpTo50.ipynb`. It imports compatible old masks,
generates only missing ones, and publishes the complete cache. H1, H2 and H3 only read that cache;
they do not install or run SAM. Missing masks stop this notebook with instructions to finish preparation.

**Start:** keep all four notebooks and their source ZIP in `MyDrive/Point3_Hypothesis54`.
Select a separate GPU runtime for each, using matching GPU models/software for a combined comparison.
Only mask preparation needs `HF_TOKEN` with SAM 3 access. All three experiments can use a token for
VGGT downloads, but do not require SAM access. G4 uses a separate CUDA 12.8 environment and must pass
the same numerical reference gate; GPU speedups are unmeasured.
Run cells in order. Keep the same run label when resuming. Do not run two copies of this same hypothesis
into the same output folder. After mask preparation, hypotheses can run in any order.

Forth Bridge uses the documented, user-authorized **bridge** anchor. Numeric mask checks and an all-view
overlay are saved; visual mask review is still necessary before architectural claims.


In [ ]:
import hashlib, json, os, subprocess, sys, time, zipfile
from pathlib import Path
_previous_source = globals().get('SOURCE')
from google.colab import drive
drive.mount('/content/drive')

SHARED = Path('/content/drive/MyDrive/Point3_Hypothesis54')
SOURCE_SHA256 = '83796943ffafa29f96a5fee1f4e58887a8b66b3da04515ebbd1375769ac4b446'
BUNDLE = SHARED / 'point3_shared54_source_83796943ffaf.zip'
CALIBRATION_SIZE = 18  # 18 calibration + 36 confirmation; 54 sweeps are optional.
RUN_LABEL = 'v1'       # Keep unchanged to resume; use a new label for a new study.
SESSION_HOURS = 8      # Session limit, NOT a predicted completion time.
DEVICES = '0'          # Use '0,1' only when two matching GPUs actually exist.
HYPOTHESIS = 'H1'
STUDY_ROOT = SHARED / 'results' / f'hypotheses54_{SOURCE_SHA256[:12]}_cal{CALIBRATION_SIZE}_{RUN_LABEL}'
OUTPUT = STUDY_ROOT / HYPOTHESIS
OUTPUT.mkdir(parents=True, exist_ok=True)
SOURCE = Path('/content') / ('point3_shared54_source_' + SOURCE_SHA256[:12])
if not BUNDLE.exists():
    raise FileNotFoundError(f'Keep the source ZIP in {SHARED} alongside this notebook: {BUNDLE.name}')
if hashlib.sha256(BUNDLE.read_bytes()).hexdigest() != SOURCE_SHA256:
    raise ValueError('Source bundle checksum mismatch')
with zipfile.ZipFile(BUNDLE) as archive:
    if archive.testzip() is not None: raise ValueError('Damaged source bundle')
    for entry in archive.infolist():
        dest = (SOURCE / entry.filename).resolve()
        if not dest.is_relative_to(SOURCE.resolve()): raise ValueError('Unsafe bundle path')
        payload = archive.read(entry)
        if dest.exists() and dest.read_bytes() != payload:
            raise ValueError(f'Local source changed: {dest}; use a separate version directory')
        dest.parent.mkdir(parents=True, exist_ok=True)
        if not dest.exists(): dest.write_bytes(payload)
STUDY = SOURCE / 'experiments/point3/shared54'
if _previous_source is not None and Path(_previous_source).resolve() != SOURCE.resolve():
    for _name, _module in list(sys.modules.items()):
        _file = getattr(_module, '__file__', None)
        if _file and Path(_file).resolve().is_relative_to(Path(_previous_source).resolve()):
            sys.modules.pop(_name, None)
sys.path[:0] = [str(STUDY), str(STUDY.parent), str(SOURCE)]
WORK = Path('/content/adaptiveattend_point3')
EVAL_PY = WORK / 'envs/vggt/bin/python'
SAM_PY = WORK / 'envs/sam3/bin/python'
SCRATCH = Path('/content/point3_hypothesis54_work') / HYPOTHESIS
DEADLINE = time.monotonic() + SESSION_HOURS*3600
print('Verified source:', SOURCE_SHA256[:12])
print('Persistent results:', OUTPUT)
print(HYPOTHESIS, ': 54 landmarks; up to 50 eligible distinct photographs per landmark; actual counts recorded.')

COMMON_MASKS = SHARED / 'common_masks'
MASK_ROLE = 'read'
print('Common masks:', COMMON_MASKS)
print('Mask role:', MASK_ROLE, '(all hypotheses reuse the preparation notebook outputs)')


In [ ]:
from protocol import make_plan, job_budget
PLAN = make_plan(CALIBRATION_SIZE, HYPOTHESIS)
print(json.dumps(job_budget(PLAN), indent=2))
print('Anchor types with sufficient calibration support:', PLAN['supported_anchor_types'])
print('Global-scalar fallback types:', PLAN['fallback_anchor_types'])
print('Manual override:', PLAN['anchor_overrides'])
print('Calibration IDs:', [r['scene'] for r in PLAN['rows'] if r['split']=='calibration'])

from mask_cache import require_ready
READY_MASKS = require_ready(COMMON_MASKS, PLAN)
print('Shared masks ready:', READY_MASKS['landmarks'], 'landmarks /', READY_MASKS['images'], 'masks')


## Prepare environments

Each hypothesis prepares VGGT only and reads the shared masks.
Compatible environments are reused. Credentials are read from Secrets only when launching work.


In [ ]:
from gpu_setup import gpu_stack
capabilities = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'], text=True, timeout=15).strip().splitlines()
GPU_STACK = gpu_stack(capabilities)
if GPU_STACK['blackwell']:
    EVAL_PY = WORK/'envs/vggt_cu128/bin/python'
    SAM_PY = WORK/'envs/sam3_cu128/bin/python'
print('GPU library profile:', GPU_STACK)
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True, timeout=15))
from protocol import save
save(OUTPUT/'gpu_setup.json', GPU_STACK)

from hypothesis_study import bounded_run, BudgetExceeded, write_json

def checked(command, label):
    print(label, flush=True)
    return bounded_run(command, OUTPUT / 'setup.log', SOURCE, os.environ.copy(), DEADLINE)

def pinned_repository(url, directory, revision):
    directory.parent.mkdir(parents=True, exist_ok=True)
    if not directory.exists():
        checked(['git', 'clone', '--quiet', url, directory], f'Cloning {directory.name}')
    # Read-only inspection; never overwrite a checkout with local edits.
    status = subprocess.check_output(['git', '-C', str(directory), 'status', '--porcelain'], text=True, timeout=10)
    if status.strip():
        raise RuntimeError(f'Local edits found in {directory}; resolve them before changing its revision')
    checked(['git', '-C', directory, 'fetch', '--quiet', 'origin', revision], f'Pinning {directory.name}')
    checked(['git', '-C', directory, 'checkout', '--quiet', revision], f'Checking {directory.name} revision')

eval_probe = ("import torch,numpy,safetensors,sys,importlib;"
              f"assert torch.__version__.split('+')[0]=={GPU_STACK['eval_torch']!r},torch.__version__;"
              "assert numpy.__version__=='2.2.6',numpy.__version__;"
              f"assert torch.version.cuda=={GPU_STACK['eval_cuda']!r},torch.version.cuda;"
              "assert torch.cuda.is_available(),'Select a GPU runtime';torch.ones(2,device='cuda').sum().item();"
              "sys.path[:0]=['intervention','.'];importlib.import_module('models.vggt.run_pass')")
sam_probe = ("import torch,decord,pycocotools;from sam3.model_builder import build_sam3_image_model;"
             "from sam3.model.sam3_image_processor import Sam3Processor;"
             "assert torch.__version__.startswith('2.7.0'),torch.__version__;"
             f"assert torch.version.cuda=={GPU_STACK['sam_cuda']!r},torch.version.cuda;"
             "assert torch.cuda.is_available(),'SAM requires a GPU';torch.ones(2,device='cuda').sum().item()")

def ready(python, probe, repo, revision):
    if not python.exists() or not (repo / '.git').exists():
        return False
    head = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True, timeout=10).strip()
    dirty = subprocess.check_output(['git', '-C', str(repo), 'status', '--porcelain'], text=True, timeout=10).strip()
    if head != revision or dirty:
        return False
    try:
        checked([python, '-c', probe], f'Checking existing {python.parent.parent.name} environment')
        return True
    except BudgetExceeded:
        raise
    except RuntimeError:
        return False

vggt_revision = '97bbde571faddde3ace3cfa7724a20448026c4c8'
sam_revision = '11dec2936de97f2857c1f76b66d982d5a001155d'
vggt_dir, sam_dir = WORK / 'third_party/vggt', WORK / 'third_party/sam3'
try:
    eval_ready = ready(EVAL_PY, eval_probe, vggt_dir, vggt_revision)
    sam_ready = True
    if not (eval_ready and sam_ready):
        checked([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], 'Preparing isolated environments')
        uv = [sys.executable, '-m', 'uv']
        if not eval_ready:
            if not EVAL_PY.exists():
                checked(uv + ['venv', '--python', '3.12', EVAL_PY.parent.parent], 'Creating VGGT environment')
            pip = uv + ['pip', 'install', '--python', EVAL_PY]
            checked(pip + ['torch=='+GPU_STACK['eval_torch'], 'torchvision=='+GPU_STACK['eval_vision'], '--index-url', 'https://download.pytorch.org/whl/cu'+GPU_STACK['eval_cuda'].replace('.','')], 'Installing VGGT GPU libraries')
            checked(pip + ['numpy==2.2.6', 'safetensors', '-r', SOURCE / 'requirements.txt'], 'Installing evaluation dependencies')
            pinned_repository('https://github.com/facebookresearch/vggt.git', vggt_dir, vggt_revision)
            checked(pip + ['--no-deps', '-e', vggt_dir], 'Installing VGGT')
        if not sam_ready:
            if not SAM_PY.exists():
                checked(uv + ['venv', '--python', '3.12', SAM_PY.parent.parent], 'Creating SAM 3 environment')
            pip = uv + ['pip', 'install', '--python', SAM_PY]
            checked(pip + ['setuptools<81', 'wheel'], 'Installing SAM build dependencies')
            checked(pip + ['torch==2.7.0', 'torchvision==0.22.0', '--index-url', 'https://download.pytorch.org/whl/cu'+GPU_STACK['sam_cuda'].replace('.','')], 'Installing SAM GPU libraries')
            pinned_repository('https://github.com/facebookresearch/sam3.git', sam_dir, sam_revision)
            checked(pip + ['-e', sam_dir, 'einops', 'psutil', 'numpy<2', 'pillow', 'decord==0.6.0', 'pycocotools'], 'Installing SAM 3 dependencies')
    checked([EVAL_PY, '-c', eval_probe], 'Validating VGGT imports and GPU')
    pass # Experiment notebooks only read masks.
    # Preserve actual package versions and include them in checkpoint identity.
    for name, python in [('evaluation', EVAL_PY)]:
        versions_path = OUTPUT / (name+'_packages.json')
        script = ("import importlib.metadata,json,pathlib; "
                  f"pathlib.Path({str(versions_path)!r}).write_text(json.dumps(sorted((d.metadata['Name'],d.version) for d in importlib.metadata.distributions())))")
        bounded_run([python, '-c', script], OUTPUT / 'setup.log', SOURCE, os.environ.copy(), DEADLINE)
    print(f'Environments ready. {(DEADLINE-time.monotonic())/60:.1f} minutes remain in this session.')
except Exception as exc:
    write_json(OUTPUT / 'status.json', {'status': 'setup_incomplete', 'reason': type(exc).__name__, 'num_views': 50})
    raise


In [ ]:
def run_phase(phase):
    from google.colab import userdata
    from hypothesis_study import bounded_run
    remaining = (DEADLINE-time.monotonic())/3600
    if remaining <= 0:
        raise RuntimeError('Session budget expired. Rerun settings to start a new session and resume.')
    environment = os.environ.copy()
    try:
        try:
            environment['HF_TOKEN'] = userdata.get('HF_TOKEN')
        except Exception:
            environment.pop('HF_TOKEN', None)
            print('HF_TOKEN unavailable; using public VGGT downloads and shared masks.')
        environment['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '0'
        command = [EVAL_PY, '-u', STUDY/'run.py', '--output', OUTPUT, '--work', SCRATCH,
            '--sam-python', SAM_PY, '--phase', phase, '--hours', str(remaining),
            '--calibration-size', str(CALIBRATION_SIZE), '--devices', DEVICES,
            '--hypothesis', HYPOTHESIS, '--mask-cache', COMMON_MASKS, '--mask-role', MASK_ROLE]
        bounded_run(command, OUTPUT/'notebook.log', SOURCE, environment, DEADLINE, echo=True)
    finally:
        environment.pop('HF_TOKEN', None)


## Pilot for H1

Runs one calibration landmark with this notebook's 5 conditions.
A fresh scalar reference must match the accelerated run, including a reused model and cached inputs.
Failure blocks the bulk run. The pilot prefers a calibration scene with the largest selected view count.
There is no image-count reduction to fit GPU memory. Smaller datasets use their recorded available count.
Each hypothesis performs
its own pilot; compatible validated checkpoints are reused on resume.


In [ ]:
PILOT_PASSED = False
run_phase('pilot')
checks = [json.loads(p.read_text()) for p in OUTPUT.glob('scenes/*/reference_check.json')]
if not any(c.get('status') == 'passed' for c in checks):
    raise RuntimeError('No successful reference check found. Inspect scene logs.')
PILOT_PASSED = True
for path in OUTPUT.glob('scenes/*/scene_timing.json'):
    print(path.parent.name, json.loads(path.read_text()))
print(HYPOTHESIS, 'pilot passed. Scene timing is not a dataset-wide runtime guarantee.')


## Run or resume H1

Finish H1 calibration, freeze its choices, then evaluate all 36 confirmation landmarks.
This cell does not start the other hypotheses or wait for their reconstruction results.
The preparation notebook must have completed the shared masks before this run starts.
Validated checkpoints are reused after a timeout or disconnection. Failed landmarks stay visible.


In [ ]:
if not globals().get('PILOT_PASSED', False):
    raise RuntimeError('Run and pass the pilot cell first.')
run_phase('all')


## H1 results and mask review

This notebook creates its own paired table, plot, summary and strength diagnostics.
Partial results are descriptive; they do not establish a benefit across the full confirmation cohort.
The two-point grids and one seed are preliminary experimental choices.


In [ ]:
import pandas as pd
from IPython.display import display, Markdown, Image
report_folder = OUTPUT/'reports'/HYPOTHESIS
if (report_folder/'summary.json').exists():
    display(Markdown((report_folder/'REPORT.md').read_text()))
    print(json.dumps(json.loads((report_folder/'summary.json').read_text()), indent=2))
    paired = pd.read_csv(report_folder/'paired_results.csv')
    display(paired)
    if len(paired):
        print('Mean structured − no-intervention:', paired['structured_minus_baseline'].mean())
        plot = report_folder/'paired_differences.png'
        if plot.exists(): display(Image(filename=str(plot)))
        if HYPOTHESIS == 'H3':
            display(paired.groupby(['anchor','anchor_fallback'])['structured_minus_scalar'].agg(['count','mean','median']))
    display(pd.read_csv(OUTPUT/'failures.csv'))
    display(pd.read_csv(OUTPUT/'strength_diagnostics.csv'))
else: print('No report yet. Inspect notebook.log and scene failure logs.')
bridge = hashlib.sha256('Forth_Bridge'.encode()).hexdigest()[:16]
preview = OUTPUT/'scenes'/bridge/'mask_preview.jpg'
if preview.exists():
    display(Markdown('### Forth Bridge — manual bridge anchor; review every selected-view overlay'))
    display(Image(filename=str(preview)))
    print(json.loads((preview.parent/'mask_validation.json').read_text()))
else: print('Forth Bridge has not yet run here; review its masks in the preparation notebook.')
print('Saved results:', OUTPUT)
reuse = [json.loads(p.read_text()) for p in OUTPUT.glob('scenes/*/mask_reuse.json')]
if reuse:
    print('Mask totals:', {key: sum(r.get(key, 0) for r in reuse) for key in ['reused','imported','generated']})
print('Common reusable mask cache:', COMMON_MASKS)
